# Spain Power System — DC OPF Results

Reads the CSV files saved by `run_opf.jl` from the `results/` directory and produces:
- OPF summary statistics
- Fuel-mix dispatch bar chart
- Installed capacity vs. dispatch comparison
- Generator dispatch scatter (capacity vs. dispatch, coloured by fuel)
- Branch loading distribution
- Congested branches

**Run `julia run_opf.jl` first** to generate the result files.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

RESULTS = Path('results')

# Colour palette per fuel type
FUEL_COLORS = {
    'Wind':    '#4CAF50',
    'Solar':   '#FFC107',
    'Hydro':   '#2196F3',
    'Nuclear': '#9C27B0',
    'Gas':     '#F44336',
    'Coal':    '#212121',
    'Biomass': '#795548',
    'Oil':     '#607D8B',
    'Slack':   '#BDBDBD',
}

In [ ]:
# Load result CSVs
def load_results():
    missing = [f for f in ['summary.csv', 'gen_dispatch.csv', 'fuel_mix.csv', 'branch_flows.csv']
               if not (RESULTS / f).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing result files: {missing}\n"
            "Run  julia run_opf.jl  first to generate them."
        )
    return (
        pd.read_csv(RESULTS / 'summary.csv'),
        pd.read_csv(RESULTS / 'gen_dispatch.csv'),
        pd.read_csv(RESULTS / 'fuel_mix.csv'),
        pd.read_csv(RESULTS / 'branch_flows.csv'),
    )

summary_df, gen_df, fuel_df, branch_df = load_results()
print('Result files loaded successfully.')

## Summary

In [ ]:
s = summary_df.iloc[0]
print('=' * 50)
print('DC OPF Summary')
print('=' * 50)
print(f"  Status            : {s['status']}")
print(f"  Total cost        : {s['objective_eur_h']:,.0f} €/h")
print(f"  Total generation  : {s['total_gen_mw']:,.1f} MW")
print(f"  Total load        : {s['total_load_mw']:,.1f} MW")
print(f"  Mismatch          : {s['mismatch_mw']:+.2f} MW")
n_congested = (branch_df['loading_pct'] > 90).sum()
print(f"  Congested branches: {n_congested}  (>90% of thermal limit)")
print('=' * 50)

## Fuel Mix Dispatch

In [ ]:
active = fuel_df[fuel_df['dispatch_mw'] > 1].sort_values('dispatch_mw', ascending=True)
colors = [FUEL_COLORS.get(f, '#90A4AE') for f in active['fuel']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Horizontal bar — dispatch
bars = axes[0].barh(active['fuel'], active['dispatch_mw'] / 1e3, color=colors)
for bar, val in zip(bars, active['dispatch_mw']):
    axes[0].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                 f"{val/1e3:.1f} GW", va='center', fontsize=9)
axes[0].set_xlabel('Dispatch (GW)')
axes[0].set_title('Fuel Mix — DC OPF Dispatch')

# Pie chart
axes[1].pie(
    active['dispatch_mw'],
    labels=active['fuel'],
    colors=colors,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.8,
)
axes[1].set_title('Generation Mix (%)')

plt.tight_layout()
plt.savefig(RESULTS / 'fuel_mix.png', bbox_inches='tight')
plt.show()

## Installed Capacity vs. Dispatch

In [ ]:
fuels_sorted = fuel_df.sort_values('dispatch_mw', ascending=False)
fuels_sorted = fuels_sorted[fuels_sorted['capacity_mw'] > 1]

x     = np.arange(len(fuels_sorted))
width = 0.38

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - width / 2, fuels_sorted['capacity_mw'] / 1e3, width,
       label='Installed Capacity', color='steelblue', alpha=0.75)
ax.bar(x + width / 2, fuels_sorted['dispatch_mw'] / 1e3, width,
       label='Dispatch', color='tomato', alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(fuels_sorted['fuel'], rotation=30, ha='right')
ax.set_ylabel('Power (GW)')
ax.set_title('Installed Capacity vs. Dispatch by Fuel')
ax.legend()

plt.tight_layout()
plt.savefig(RESULTS / 'capacity_vs_dispatch.png', bbox_inches='tight')
plt.show()

## Top Dispatched Generators

In [ ]:
top_n   = 25
top_gen = (gen_df[gen_df['dispatch_mw'] > 0.5]
           .sort_values('dispatch_mw', ascending=False)
           .head(top_n))
colors  = [FUEL_COLORS.get(f, '#90A4AE') for f in top_gen['fuel']]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(top_gen['unit_name'], top_gen['dispatch_mw'], color=colors)

# utilisation annotation
for bar, util in zip(bars, top_gen['utilization_pct']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
            f"{util:.0f}%", va='center', fontsize=8)

ax.set_xlabel('Dispatch (MW)')
ax.set_title(f'Top {top_n} Dispatched Generators (% = utilisation)')

# Legend
shown = top_gen['fuel'].unique()
legend_patches = [mpatches.Patch(color=FUEL_COLORS.get(f, '#90A4AE'), label=f) for f in shown]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS / 'top_generators.png', bbox_inches='tight')
plt.show()

## Generator Capacity vs. Dispatch Scatter

In [ ]:
plot_df = gen_df[gen_df['capacity_mw'] > 0].copy()

fig, ax = plt.subplots(figsize=(10, 7))
for fuel, grp in plot_df.groupby('fuel'):
    ax.scatter(grp['capacity_mw'], grp['dispatch_mw'],
               color=FUEL_COLORS.get(fuel, '#90A4AE'),
               alpha=0.7, s=40, label=fuel)

# 100% utilisation reference line
max_cap = plot_df['capacity_mw'].max()
ax.plot([0, max_cap], [0, max_cap], 'k--', linewidth=0.8, label='100% utilisation')

ax.set_xlabel('Installed Capacity (MW)')
ax.set_ylabel('Dispatch (MW)')
ax.set_title('Generator Capacity vs. Dispatch')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS / 'gen_scatter.png', bbox_inches='tight')
plt.show()

## Branch Loading

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loading distribution histogram
axes[0].hist(branch_df['loading_pct'], bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(90, color='red', linestyle='--', linewidth=1.5, label='90% threshold')
axes[0].set_xlabel('Loading (%)')
axes[0].set_ylabel('Number of branches')
axes[0].set_title('Branch Loading Distribution')
axes[0].legend()

# Congested branches
congested = (branch_df[branch_df['loading_pct'] > 90]
             .sort_values('loading_pct', ascending=True)
             .tail(20))
if len(congested) > 0:
    cmap = plt.cm.YlOrRd
    norm = plt.Normalize(90, min(congested['loading_pct'].max(), 150))
    bar_colors = [cmap(norm(v)) for v in congested['loading_pct']]
    axes[1].barh(congested['branch_name'], congested['loading_pct'], color=bar_colors)
    axes[1].axvline(90,  color='red',    linestyle='--', linewidth=1, label='90%')
    axes[1].axvline(100, color='darkred', linestyle='-',  linewidth=1, label='100%')
    axes[1].set_xlabel('Loading (%)')
    axes[1].set_title(f'Top Congested Branches (n={len(congested)})')
    axes[1].legend(fontsize=9)
else:
    axes[1].text(0.5, 0.5, 'No congested branches\n(all below 90%)',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=12)
    axes[1].set_title('Congested Branches')
    axes[1].axis('off')

plt.tight_layout()
plt.savefig(RESULTS / 'branch_loading.png', bbox_inches='tight')
plt.show()

## Marginal Cost by Fuel

In [ ]:
dispatched = gen_df[gen_df['dispatch_mw'] > 0.5].copy()

fig, ax = plt.subplots(figsize=(11, 5))
for fuel, grp in dispatched.groupby('fuel'):
    ax.scatter(grp['cost_eur_per_mwh'], grp['dispatch_mw'],
               color=FUEL_COLORS.get(fuel, '#90A4AE'),
               alpha=0.75, s=50, label=fuel)

ax.set_xlabel('Marginal cost (€/MWh)')
ax.set_ylabel('Dispatch (MW)')
ax.set_title('Dispatch vs. Marginal Cost (merit-order view)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS / 'merit_order.png', bbox_inches='tight')
plt.show()